# Notebook to identify bottom 5% residues showing low PAE
Using each protein's numpy arrays for PAE and its standard deviation (sigma), we try to identify the residues that show the high confidence (low PAE) with low variance (low sigma)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

In [ ]:
protein_data = {
    "Lsr2": "P9WIP7", "EspD": "P9WJD5", "EspA": "P9WJE1", "EspM": "P96214",
    "EspE": "P9WJD3", "EspH": "O69732", "PPE68": "P9WHW9", "EspI": "P9WJC5",
    "EspJ": "P9WJC3", "EspK": "P9WJC1", "EspB": "P9WJD9"
}

idr_data = {
    "Lsr2": [(61, 76)],
    "EspD": [(1, 14), (35, 60), (171, 184)],
    "EspA": [(270, 294), (301, 383)],
    "EspM": [(1, 34), (131, 161), (230, 249)],
    "EspE": [(204, 277), (361, 402)],
    "EspH": [(1, 33)],
    "PPE68": [(173, 202), (236, 368)],
    "EspI": [(1, 246), (257, 311), (325, 373)],
    "EspJ": [(97, 113), (176, 280)],
    "EspK": [(175, 461)],
    "EspB": [(93, 124), (265, 335), (352, 433), (453, 460)]
}

motif_data = {
    'PPE68': [(341, 343), (265, 267)],
    'EspA': [(346, 348), (279, 281)],
    'EspB': [(456, 460), (456, 460), (459, 460), (384, 386)],
    'EspE': [(368, 370)],
    'EspI': [(57, 59), (206, 208), (217, 219), (286, 288), (351, 353), (1, 4)],
    'EspJ': [(196, 198), (257, 259), (102, 104)],
    'EspK': [(252, 254), (329, 331)],
    'EspM': [(146, 148), (246, 248), (1, 4)]
}

MATRIX_FOLDER = "pae_sigma_matrices"

In [ ]:
# Reverse mapping to look up protein name via UniProt ID as that's how they're saved
uniprot_to_name = {v: k for k, v in protein_data.items()}
all_protein_records = []
# Collect all PAE and sigma values across all proteins for global thresholds
all_pae_vals = []
all_sigma_vals = []
for uniprot_id, protein_name in uniprot_to_name.items():
    pae_path = os.path.join(MATRIX_FOLDER, f"{uniprot_id}_PAE.npy")
    sigma_path = os.path.join(MATRIX_FOLDER, f"{uniprot_id}_sigma.npy")
    if not (os.path.exists(pae_path) and os.path.exists(sigma_path)):
        continue
    pae = np.load(pae_path)
    sigma = np.load(sigma_path)
    nres = pae.shape[0]
    # Exclude diagonal (self-interactions)
    off_diag = ~np.eye(nres, dtype=bool)
    all_pae_vals.append(pae[off_diag].ravel())
    all_sigma_vals.append(sigma[off_diag].ravel())
# Global thresholds: 5th percentile of pooled PAE and sigma distributions
global_pae_threshold = np.percentile(np.concatenate(all_pae_vals), 5)
global_sigma_threshold = np.percentile(np.concatenate(all_sigma_vals), 5)
print(f"Global PAE 5th percentile threshold: {global_pae_threshold:.4f}")
print(f"Global sigma 5th percentile threshold: {global_sigma_threshold:.4f}")
# Retrieve matrix files for per protein analyses
for uniprot_id, protein_name in uniprot_to_name.items():
    pae_path = os.path.join(MATRIX_FOLDER, f"{uniprot_id}_PAE.npy")
    sigma_path = os.path.join(MATRIX_FOLDER, f"{uniprot_id}_sigma.npy")
    if not (os.path.exists(pae_path) and os.path.exists(sigma_path)):
        print(f"Skipping {protein_name} ({uniprot_id}): Matrices not found.")
        continue
    pae = np.load(pae_path)
    sigma = np.load(sigma_path)
    nres = pae.shape[0]
    residues_1based = np.arange(1, nres + 1)
    # Build "confident contact" boolean matrix using global thresholds
    confident = (pae <= global_pae_threshold) & (sigma <= global_sigma_threshold)
    # Zero out diagonal so a residue doesn't count itself
    np.fill_diagonal(confident, False)
    idr_regions = idr_data.get(protein_name, [])
    is_idr_mask = np.zeros(nres, dtype=bool)
    for start, end in idr_regions:
        is_idr_mask[(residues_1based >= start) & (residues_1based <= end)] = True
    idr_indices = np.where(is_idr_mask)[0]
    non_idr_indices = np.where(~is_idr_mask)[0]
    n_idr_total = len(idr_indices)
    n_non_idr_total = len(non_idr_indices)
    n_all_total = nres - 1  # exclude self
    protein_records = []
    for idx in range(nres):
        res_num = residues_1based[idx]
        in_idr = is_idr_mask[idx]
        # Flag residue in motif
        in_motif = False
        current_motifs = motif_data.get(protein_name, [])
        for m_start, m_end in current_motifs:
            if m_start <= res_num <= m_end:
                in_motif = True
                break
        # Count confident contacts in each subset
        # All residues
        n_confident_all = int(confident[idx].sum())
        # IDR residues only (exclude self if residue is in IDR)
        if n_idr_total > 0:
            idr_targets = idr_indices[idr_indices != idx] if in_idr else idr_indices
            n_confident_idr = int(confident[idx, idr_targets].sum())
            denom_idr = len(idr_targets)
        else:
            n_confident_idr = 0
            denom_idr = 0
        # Non-IDR residues only (exclude self if residue is not in IDR)
        if n_non_idr_total > 0:
            non_idr_targets = non_idr_indices[non_idr_indices != idx] if not in_idr else non_idr_indices
            n_confident_non_idr = int(confident[idx, non_idr_targets].sum())
            denom_non_idr = len(non_idr_targets)
        else:
            n_confident_non_idr = 0
            denom_non_idr = 0
        protein_records.append({
            'protein_name': protein_name,
            'uniprot_id': uniprot_id,
            'residue_number': res_num,
            'is_in_idr': in_idr,
            'is_in_motif': in_motif,
            'n_confident_contacts_all': n_confident_all,
            'n_confident_contacts_idr': n_confident_idr,
            'n_confident_contacts_non_idr': n_confident_non_idr,
        })
    df_temp = pd.DataFrame(protein_records)
    all_protein_records.append(df_temp)
master_df = pd.concat(all_protein_records, ignore_index=True)
master_df['global_pae_threshold'] = global_pae_threshold
master_df['global_sigma_threshold'] = global_sigma_threshold
final_cols = [
    'protein_name', 'uniprot_id', 'residue_number', 'is_in_idr', 'is_in_motif',
    'n_confident_contacts_all', 'n_confident_contacts_idr', 'n_confident_contacts_non_idr',
    'global_pae_threshold', 'global_sigma_threshold',
]
master_df = master_df[final_cols]
# Round fractional columns
frac_cols = ['global_pae_threshold', 'global_sigma_threshold']
master_df[frac_cols] = master_df[frac_cols].astype(float).round(4)
master_df.to_csv("master_residues_confident_contacts.csv", index=False)
print("Analysis complete! Master file saved as 'master_residues_confident_contacts.csv'.")

Global PAE 5th percentile threshold: 0.2500
Global sigma 5th percentile threshold: 0.0252
Analysis complete! Master file saved as 'master_residues_confident_contacts.csv'.
